# v00 · First Age text hyperindex — mechanism
`PtolemyDesktop/Archive/ainulindale_first_age/FA_smnnip_hyperindex.py` (in-file date: April 2026; git first commit 2026-04-18). Base-97 mixed-radix address + Fano base-7 path index, length stored separately.

In [1]:
import importlib.util, os, sys, time, math, hashlib, json
from collections import Counter
PROJ = os.environ.get("PROJ", "/home/rendier/Projects/ThePlace")
NB = os.path.join(PROJ, "FourthAgePapers/HyperindexingSystem/notebooks")
CORPUS = open(os.path.join(NB, "corpus_sample.txt"), encoding="utf-8").read()   # fixed 4000-char sample, 97-symbol keyboard set
def load(rel, name="m"):
    """Load a source file from the repo by path, unmodified."""
    spec = importlib.util.spec_from_file_location(name, os.path.join(PROJ, rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
print("corpus:", len(CORPUS), "chars, sha256", hashlib.sha256(CORPUS.encode()).hexdigest()[:16])

corpus: 4000 chars, sha256 3105964b9180566a


## Load the file unmodified and run its own record function

In [2]:
fa = load("PtolemyDesktop/Archive/ainulindale_first_age/FA_smnnip_hyperindex.py", "fa")
T = fa.TextHyperIndex
seq = "Hello, World!"
text_idx, fano_idx, n = T.index_record(seq)
print("text_index:", text_idx)
print("fano_index:", fano_idx)
print("length    :", n)
assert T.decode_text(text_idx, n) == seq and T.verify(seq)

text_index: 22926229613755264751551883
fano_index: 78409984071
length    : 13


## The charset order is `a–z A–Z 0–9 punctuation space tab newline`
This is the order later carried into `ValaQuenta/modules/hyperwebster`.

In [3]:
print(fa.US_KEYBOARD_CHARS[:26], "...", repr(fa.US_KEYBOARD_CHARS[-3:]))
print("index of 'a':", fa.USKeyboardMap.encode("a"), "| space:", fa.USKeyboardMap.encode(" "), "| newline:", fa.USKeyboardMap.encode("\n"))

abcdefghijklmnopqrstuvwxyz ... ' \t\n'
index of 'a': 0 | space: 94 | newline: 96


## This is not a bijection over strings: length is part of the address
Character `a` has index 0, so every string of `a`s has integer 0. The stored `length` is what tells them apart. (The later versions v02 onward use *bijective* base-N and drop this need.)

In [4]:
ids = {k: T.encode_text("a"*k) for k in (1, 2, 3, 10)}
print(ids)
assert set(ids.values()) == {0}
# distinct strings of the same length are still distinct
assert T.encode_text("ab") != T.encode_text("ba")

{1: 0, 2: 0, 3: 0, 10: 0}


## The Fano index is many-to-one by construction (`char_index % 7`)
Seven classes only; measured on the corpus sample: how many distinct 4-character windows share a Fano index.

In [5]:
wins = {CORPUS[i:i+4] for i in range(len(CORPUS)-3)}
by_fano = {}
for w in wins: by_fano.setdefault(T.encode_fano(w), set()).add(w)
print("distinct 4-windows:", len(wins), "| distinct Fano indices:", len(by_fano), "| max windows sharing one Fano index:", max(len(v) for v in by_fano.values()))
assert len(by_fano) < len(wins)

distinct 4-windows: 1908 | distinct Fano indices: 1061 | max windows sharing one Fano index: 10
